# Stage 2: Instruction Fine-Tuning
## Domain: E-commerce Product Support Assistant

**Goal:** Teach the model to follow instructions and answer e-commerce support questions
clearly, using a curated instruction-response dataset (`data/instruction_dataset.jsonl`,
100 examples covering cancellations, returns, refunds, delivery tracking, gift cards,
damaged items, firmware issues, marketplace orders, account recovery, and subscriptions).

We continue training from the Stage 1 merged model (domain-adapted) using LoRA SFT.


## 1. Install required libraries

In [1]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new]"
!pip install trl peft accelerate bitsandbytes


## 2. Load tokenizer & 3. Load model (continuing from Stage 1)

In [2]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

# Use the Stage 1 merged model if available, otherwise fall back to the base model.
STAGE1_MERGED_DIR = "/content/stage1_non_instruction_merged_model"
BASE_MODEL = "unsloth/Qwen2.5-0.5B-bnb-4bit"

import os
model_path = STAGE1_MERGED_DIR if os.path.exists(STAGE1_MERGED_DIR) else BASE_MODEL

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. Formatting the instruction dataset

In [3]:
import json
from datasets import load_dataset

# Upload data/instruction_dataset.jsonl to /content/ before running this cell
dataset = load_dataset("json", data_files="/content/instruction_dataset.jsonl", split="train")
print(dataset)
print(dataset[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'source_page', 'topic'],
    num_rows: 100
})
{'instruction': 'Can a customer cancel an order after payment has been made?', 'input': 'A customer contacts support saying they just paid for an order and want to cancel it immediately.', 'output': 'Order cancellation after payment depends on the warehouse processing status. If the warehouse has not yet packed the order, cancellation is typically possible. However, if the item has already been packed or dispatched, cancellation may not be guaranteed. Support agents should check the order status in real time and advise the customer accordingly, noting that outcomes vary based on exact timing.', 'source_page': 1, 'topic': 'Order cancellation'}


In [4]:
PROMPT_TEMPLATE = '''### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''

PROMPT_TEMPLATE_NO_INPUT = '''### Instruction:
{instruction}

### Response:
{output}'''

EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    if example.get("input"):
        text = PROMPT_TEMPLATE.format(
            instruction = example["instruction"],
            input = example["input"],
            output = example["output"],
        )
    else:
        text = PROMPT_TEMPLATE_NO_INPUT.format(
            instruction = example["instruction"],
            output = example["output"],
        )
    return {"text": text + EOS_TOKEN}

dataset = dataset.map(format_example)
print(dataset[0]["text"])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

### Instruction:
Can a customer cancel an order after payment has been made?

### Input:
A customer contacts support saying they just paid for an order and want to cancel it immediately.

### Response:
Order cancellation after payment depends on the warehouse processing status. If the warehouse has not yet packed the order, cancellation is typically possible. However, if the item has already been packed or dispatched, cancellation may not be guaranteed. Support agents should check the order status in real time and advise the customer accordingly, noting that outcomes vary based on exact timing.<|endoftext|>


## 5. Applying LoRA

In [5]:
STAGE2_RANK = 16
STAGE2_ALPHA = 16
STAGE2_DROPOUT = 0.0
STAGE2_LR = 1e-4
STAGE2_MAX_STEPS = 90
STAGE2_BATCH_SIZE = 2

model = FastLanguageModel.get_peft_model(
    model,
    r = STAGE2_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = STAGE2_ALPHA,
    lora_dropout = STAGE2_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


Unsloth 2026.6.9 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 6. Training the model

In [6]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,                 # preserve prompt/response boundaries
    args = SFTConfig(
        per_device_train_batch_size = STAGE2_BATCH_SIZE,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = STAGE2_MAX_STEPS,
        learning_rate = STAGE2_LR,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/stage2_outputs",
    ),
)

stage2_stats = trainer.train()
print(stage2_stats)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 7 | Total steps = 90
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,3.007281
10,2.772389
15,2.452176
20,2.399072
25,2.248429
30,2.229714
35,2.083346
40,2.061143
45,2.047033
50,1.895282


Unsloth: Restored added_tokens_decoder metadata in /content/stage2_outputs/checkpoint-90/tokenizer_config.json.


TrainOutput(global_step=90, training_loss=2.0854831377665204, metrics={'train_runtime': 168.4338, 'train_samples_per_second': 4.275, 'train_steps_per_second': 0.534, 'total_flos': 189677338483200.0, 'train_loss': 2.0854831377665204, 'epoch': 6.96})


## 7. Saving adapter / model

In [7]:
STAGE2_ADAPTER_DIR = "/content/stage2_instruction_adapter"
model.save_pretrained(STAGE2_ADAPTER_DIR)
tokenizer.save_pretrained(STAGE2_ADAPTER_DIR)
print("Saved Stage 2 adapter to:", STAGE2_ADAPTER_DIR)

STAGE2_MERGED_DIR = "/content/stage2_instruction_merged_model"
model.save_pretrained_merged(STAGE2_MERGED_DIR, tokenizer, save_method = "merged_16bit")
print("Saved Stage 2 merged model to:", STAGE2_MERGED_DIR)

# Optional: push to Hugging Face Hub
# model.push_to_hub_merged("shahmitul1809/ecommerce-support-sft", tokenizer, save_method="merged_16bit", token="hf_xxx")


Unsloth: Restored added_tokens_decoder metadata in /content/stage2_instruction_adapter/tokenizer_config.json.


Saved Stage 2 adapter to: /content/stage2_instruction_adapter


config.json:   0%|          | 0.00/774 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/stage2_instruction_merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:08<00:00,  8.24s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:04<00:00,  4.66s/it]


Unsloth: Merge process complete. Saved to `/content/stage2_instruction_merged_model`
Saved Stage 2 merged model to: /content/stage2_instruction_merged_model


## 8. Running inference after instruction fine-tuning

In [8]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=150):
    prompt = f"### Instruction:\n{question}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text.split("### Response:")[-1].strip()

print(ask("How can I apply for reimbursement after returning a damaged item?"))


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


If you return a damaged item and the manufacturer or seller cannot identify the original packaging or condition, you may be eligible for reimbursement. You should contact the seller or manufacturer directly to confirm the condition of the item and request a return. The seller or manufacturer may need to send you a replacement or refund. If the item is not damaged beyond repair, you may be eligible for a full refund.


## 10 Domain Questions for Evaluation

In [9]:
EVAL_QUESTIONS = [
    "Can a customer cancel an order after payment has been made?",
    "Why does my order show 'shipped' but tracking has not updated for two days?",
    "Can I exchange a gift card balance for cash?",
    "What should I do with a damaged package before requesting a refund?",
    "My new electronic device seems faulty right out of the box, what should I check first?",
    "I ordered two items in one cart but only one item ships today, why?",
    "How long does a refund take if I paid by credit card?",
    "I forgot my password and the verification email is not in my inbox, what should I do?",
    "How can I cancel a subscription before it renews automatically?",
    "I removed the tags from a clothing item, can I still return it?",
]

for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"Q{i}: {q}")
    print("A:", ask(q))
    print("-" * 80)


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: Can a customer cancel an order after payment has been made?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: If a customer has already placed an order and is cancelling it, the platform should not automatically cancel the order. The customer must be prompted to cancel the order before the payment is fully processed. The platform should also verify the cancellation request and ensure that the customer has not already placed a new order. Support should be available to guide the customer through the cancellation process.
--------------------------------------------------------------------------------
Q2: Why does my order show 'shipped' but tracking has not updated for two days?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: Credit card refunds typically take 2–3 business days to process. The bank sends the refund to the cardholder's bank account within that timeframe. If the card has not yet been debited, the refund may not appear on the card until the bank credits it. The refund status is usually displayed on the card's statement or online.
--------------------------------------------------------------------------------
Q8: I forgot my password and the verification email is not in my inbox, what should I do?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: If you forgot your password and the verification email is not in your inbox, you should follow these steps: 1. Check your spam or junk folder for the verification email. 2. If the email is not in your inbox, check your spam folder again. 3. If the email is in your inbox, click on it and look for a link to reset your password. 4. Follow the instructions in the email to reset your password. 5. If you still cannot reset your password, contact support for assistance.
--------------------------------------------------------------------------------
Q9: How can I cancel a subscription before it renews automatically?


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: If a customer has already subscribed to a product and wants to cancel before the renewal date, the platform should allow them to do so. The customer should be prompted to confirm the cancellation before the renewal date. The platform should not automatically renew the subscription for the customer. Support should be available to guide the customer through the cancellation process.
--------------------------------------------------------------------------------
Q10: I removed the tags from a clothing item, can I still return it?
A: If the tags were removed before the item was shipped, you may be eligible for a return if the item is in the original packaging and the tags were not attached at the time of purchase. The platform may verify the tags before processing the return. If the item is in the original packaging and the tags are attached, you may be able to return it.
--------------------------------------------------------------------------------
